# MiniMax H3 pruned REF2VA no Kaggle

Adaptação do workflow **Minimax H3 realistic, most realistic skin** enviado pelo usuário.
Usa **`minimax_h3_ref2va_pruned_int8_convrot`**, que preserva o condicionamento por referências;
**não** troca REF2VA pelo FL2VA mencionado em metadados antigos do JSON.

**Antes de Run All:** em Settings, selecione GPU e habilite Internet. Anexe um Dataset com sua
imagem e, de preferência, os modelos listados abaixo. Preencha `REFERENCE_IMAGES` na célula 1.
Execute as células em ordem. O vídeo aparece no próprio notebook e em `/kaggle/working/minimax_h3/output`.
Não precisa de túnel, conta RunPod, API paga nem token Discord.

O modelo pruned continua grande: aproximadamente **39 GiB / 42 GB** com o encoder, VAEs e LoRA turbo.
T4 × 2 **não vira uma GPU de 32 GB**: este notebook usa somente a GPU 0.
Em T4/P100 o encoder NVFP4 depende do fallback de desquantização, não de kernels FP4 nativos;
isso pode consumir mais memória e ser muito lento. Não há garantia de que toda sessão Kaggle comporte H3.
O notebook interrompe downloads que não cabem no disco e mostra erros de execução, sem declarar sucesso falso.

**Adaptação de memória:** uma passagem, sem upscale, referências reduzidas, VAE em CPU,
decode em tiles, cache desativado e carregamento dinâmico apoiado em disco (`--fast-disk`).
O primeiro teste é 608 × 352, ~2,33 s, com a LoRA turbo oficial de 4 passos.
A duração curta fica abaixo da faixa treinada citada pelo ComfyUI (~5–15 s); serve para testar memória,
não para avaliar a qualidade final. Use `qualidade` depois: 864 × 480, ~5,17 s, 20 passos sem LoRA de velocidade.

**Pele realista:** prompt de close-up com luz natural. A LoRA People do anexo é opcional e precisa ser
anexada como Dataset (`PEOPLE_LORA_PATH`); seu peso e compatibilidade com o modelo pruned não foram
verificados por inferência. Quando habilitada, o notebook acrescenta `r34l1sm` ao prompt.
Os perfis não reproduzem exatamente o Singularity, DualClock ou os resultados do workflow original.

**Validação:** estrutura do notebook, lógica e esquema nativo do ComfyUI são verificáveis sem GPU.
A geração completa só pode ser validada numa sessão Kaggle com GPU e os pesos.


In [ ]:
from pathlib import Path
import os, sys, json, math, shutil, subprocess, time, uuid, struct, hashlib
from datetime import datetime, timezone

# Edite estas opções. Caminhos são do Dataset anexado em /kaggle/input.
REFERENCE_IMAGES = []  # exemplo: ['/kaggle/input/minhas-referencias/retrato.png']
REFERENCE_AUDIO = ''   # opcional: um áudio de até 15 s; deixe vazio para não usar
PRESET = 'smoke'       # smoke, conservador ou qualidade
PROMPT = """Live-action cinematic close-up of the adult person in <Picture 1>.
Preserve their face, hairstyle and outfit. One continuous shot with subtle natural
head movement. Soft window daylight, realistic skin texture and pores, natural
skin tones, no beauty filter, no plastic skin, no exaggerated sharpening.
Shallow depth of field, restrained camera movement. Audio: quiet room ambience."""
SEED = 42
PEOPLE_LORA_PATH = ''   # exemplo: /kaggle/input/meus-modelos/minimax-h3-people.safetensors
PEOPLE_STRENGTH = 0.7
CUSTOM_SPEED_LORA_PATH = ''  # substitui a turbo oficial, nunca empilha duas LoRAs de velocidade
STEPS_OVERRIDE = None       # ao usar outra LoRA, escolha os passos indicados pelo autor

# Auto procura os arquivos pelo nome em /kaggle/input; não copia os pesos do Dataset.
MODEL_PATH_OVERRIDES = {}    # opcional: {'diffusion': '/kaggle/input/.../arquivo.safetensors'}
DOWNLOAD_MISSING_MODELS = True  # False exige todos os pesos nos Datasets
VERIFY_SHA256 = False          # True verifica hashes dos pesos contra metadados HF; lê ~42 GB
HF_SECRET_NAME = ''            # opcional: nome de um Secret Kaggle; nunca coloque a chave aqui
MEMORY_MODE = 'disk'           # disk (padrão) ou classic (fallback, mais RAM em CPU)
GPU_INDEX = 0
CPU_VAE = True
RESERVE_VRAM_GB = 2.0
PORT = 8188
GENERATION_TIMEOUT_MINUTES = 180

# ComfyUI v0.32.0, commit imutável com suporte nativo ao MiniMax H3.
COMFY_COMMIT = 'c2bcbecd82ec5ae66594340b395c24ef0217b238'
COMFY_REPO = 'https://github.com/Comfy-Org/ComfyUI.git'
# Se o PyTorch Kaggle for >= 2.7, mantém a instalação existente. Caso contrário,
# instala este conjunto coerente só no venv; não altera o Python global do Kaggle.
TORCH_INDEX = 'https://download.pytorch.org/whl/cu126'
TORCH_VERSION, TORCHVISION_VERSION, TORCHAUDIO_VERSION = '2.8.0', '0.23.0', '2.8.0'

ROOT = Path('/kaggle/working/minimax_h3')
INPUT_ROOT = Path('/kaggle/input')
COMFY = ROOT / 'ComfyUI'
VENV = ROOT / '.venv'
PYTHON = VENV / 'bin/python'
BASE_URL = f'http://127.0.0.1:{PORT}'
GiB = 1024 ** 3


## 2. Funções e perfis

O modo `smoke` usa a LoRA REF2V turbo de 4 passos documentada no template oficial.
`conservador` mantém 608 × 352, mas aumenta a duração para a faixa treinada e usa o modelo base.
`qualidade` aumenta a resolução. Nenhum perfil ativa uma segunda passagem.
Para reproduzir outra LoRA de velocidade do anexo, forneça o arquivo e ajuste `STEPS_OVERRIDE`;
a compatibilidade com o pruned precisa ser testada, não é inferida pelo nome.


In [ ]:
PROFILES = {
    'smoke': {'width': 608, 'height': 352, 'seconds': 2, 'steps': 4, 'turbo': True},
    'conservador': {'width': 608, 'height': 352, 'seconds': 5, 'steps': 20, 'turbo': False},
    'qualidade': {'width': 864, 'height': 480, 'seconds': 5, 'steps': 20, 'turbo': False},
}
MODEL_FILES = {
    'diffusion': {'repo': 'Comfy-Org/MiniMax-H3', 'file': 'diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', 'folder': 'diffusion_models'},
    'encoder': {'repo': 'Comfy-Org/MiniMax-H3', 'file': 'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 'folder': 'text_encoders'},
    'video_vae': {'repo': 'Comfy-Org/MiniMax-H3', 'file': 'vae/minimax_h3_video_vae_int8_convrot.safetensors', 'folder': 'vae'},
    'audio_vae': {'repo': 'Comfy-Org/MiniMax-H3', 'file': 'vae/minimax_h3_audio_vae_fp32.safetensors', 'folder': 'vae'},
    'turbo': {'repo': 'Comfy-Org/MiniMax-H3', 'file': 'loras/minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors', 'folder': 'loras'},
}

def frame_count(seconds, fps=24):
    if not math.isfinite(seconds) or seconds <= 0:
        raise ValueError('A duração deve ser positiva e finita.')
    target = max(5, round(seconds * fps))
    return target + (5 - target % 17) % 17

def settings():
    if PRESET not in PROFILES:
        raise ValueError(f'PRESET inválido; use {list(PROFILES)}')
    cfg = dict(PROFILES[PRESET])
    cfg['steps'] = STEPS_OVERRIDE if STEPS_OVERRIDE is not None else cfg['steps']
    if not isinstance(cfg['steps'], int) or not 1 <= cfg['steps'] <= 100:
        raise ValueError('Passos devem ser um inteiro entre 1 e 100.')
    cfg['length'] = frame_count(cfg['seconds'])
    if CUSTOM_SPEED_LORA_PATH and STEPS_OVERRIDE is None:
        raise ValueError('Defina STEPS_OVERRIDE para a LoRA de velocidade personalizada.')
    return cfg

def run(args, **kwargs):
    return subprocess.run([str(x) for x in args], check=True, **kwargs)

def memory_limit():
    import psutil
    total = psutil.virtual_memory().total
    for path in [Path('/sys/fs/cgroup/memory.max'), Path('/sys/fs/cgroup/memory/memory.limit_in_bytes')]:
        if path.exists():
            value = path.read_text().strip()
            if value.isdecimal() and int(value) > 0:
                total = min(total, int(value))
    return total

def safetensors_check(path, expected_size=None):
    # Verifica estrutura e truncamento sem carregar tensores nem executar pickle.
    path = Path(path)
    size = path.stat().st_size
    if expected_size is not None and size != expected_size:
        raise ValueError(f'Tamanho diferente do publicado: {path.name}')
    with path.open('rb') as handle:
        prefix = handle.read(8)
        if len(prefix) != 8:
            raise ValueError(f'Arquivo truncado: {path.name}')
        header_size = struct.unpack('<Q', prefix)[0]
        if not 2 <= header_size <= min(100 * 1024 ** 2, size - 8):
            raise ValueError(f'Cabeçalho safetensors inválido: {path.name}')
        header = json.loads(handle.read(header_size))
    intervals = []
    for key, tensor in header.items():
        if key == '__metadata__':
            continue
        start, end = tensor['data_offsets']
        if not 0 <= start <= end <= size - 8 - header_size:
            raise ValueError(f'Tensor truncado em {path.name}: {key}')
        intervals.append((start, end))
    if not intervals:
        raise ValueError(f'Nenhum tensor em {path.name}')
    cursor = 0
    for start, end in sorted(intervals):
        if start != cursor:
            raise ValueError(f'Offsets não contíguos em {path.name}')
        cursor = end
    if cursor != size - 8 - header_size:
        raise ValueError(f'Dados extras ou ausentes em {path.name}')
    return size

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(16 * 1024 ** 2), b''):
            digest.update(block)
    return digest.hexdigest()

def find_model(key, spec):
    override = MODEL_PATH_OVERRIDES.get(key)
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f'Override inexistente para {key}: {path}')
        return path
    name = Path(spec['file']).name
    candidates = sorted(INPUT_ROOT.rglob(name)) if INPUT_ROOT.exists() else []
    if len(candidates) > 1:
        raise ValueError(f'Mais de um {name}. Escolha um em MODEL_PATH_OVERRIDES.')
    if candidates:
        return candidates[0]
    cached = ROOT / 'weights' / spec['file']
    return cached if cached.is_file() else None

def link_model(source, folder, name=None):
    source = Path(source).resolve(strict=True)
    safetensors_check(source)
    target = COMFY / 'models' / folder / (name or source.name)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink():
        if target.resolve() != source:
            raise FileExistsError(f'Link já aponta para outro arquivo: {target}')
    elif target.exists():
        if target.resolve() != source:
            raise FileExistsError(f'Não sobrescreverei {target}')
    else:
        target.symlink_to(source)
    return target.name

def ensure_download_space(required_bytes, free_bytes, reserve_bytes=5 * GiB):
    if required_bytes + reserve_bytes > free_bytes:
        raise RuntimeError(
            f'Download precisa de {required_bytes / GiB:.1f} GiB + {reserve_bytes / GiB:.1f} GiB de reserva; '
            f'há {free_bytes / GiB:.1f} GiB livres. Anexe os pesos como Dataset Kaggle '
            'em /kaggle/input e rode esta célula novamente. Não iniciamos o download.'
        )


## 3. Checar a sessão antes de instalar

Liste suas imagens e escolha os caminhos na célula 1. As imagens de exemplo do JSON não vieram
no upload e não são baixadas de links antigos de prévia. Nenhuma imagem é escolhida silenciosamente.


In [ ]:
if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Execute este notebook no Kaggle com GPU e Internet habilitados.')
ROOT.mkdir(parents=True, exist_ok=True)
cfg = settings()
gpu_info = subprocess.run(
    ['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True,
)
if gpu_info.returncode != 0:
    raise RuntimeError('GPU indisponível. Em Settings > Accelerator selecione uma GPU.')
print(gpu_info.stdout.strip())
gpu_rows = gpu_info.stdout.strip().splitlines()
if not isinstance(GPU_INDEX, int) or not 0 <= GPU_INDEX < len(gpu_rows):
    raise ValueError('GPU_INDEX não corresponde a uma GPU disponível.')
ram = memory_limit()
print(f'RAM limitada à sessão: {ram / GiB:.1f} GiB')
print(f'Disco livre em working: {shutil.disk_usage(ROOT).free / GiB:.1f} GiB')
print(f"Perfil {PRESET}: {cfg['width']} × {cfg['height']}, {cfg['length']} frames / {cfg['length']/24:.2f} s, {cfg['steps']} passos")
if ram < 16 * GiB:
    raise RuntimeError('Menos de 16 GiB de RAM: esta configuração H3 não é adequada à sessão.')
if ram < 32 * GiB:
    print('RAM apertada para H3: priorize modelos em Dataset, disk e smoke; ainda pode faltar memória.')
if not REFERENCE_IMAGES:
    examples = []
    for path in INPUT_ROOT.rglob('*'):
        if path.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}:
            examples.append(str(path))
            if len(examples) == 15:
                break
    print('Imagens encontradas:', json.dumps(examples, ensure_ascii=False, indent=2))
    raise ValueError('Preencha REFERENCE_IMAGES na célula 1 e execute novamente.')
if not 1 <= len(REFERENCE_IMAGES) <= 3:
    raise ValueError('Esta adaptação aceita 1–3 imagens; comece com uma para poupar memória.')
for path in [*REFERENCE_IMAGES, *([REFERENCE_AUDIO] if REFERENCE_AUDIO else []),
             *([PEOPLE_LORA_PATH] if PEOPLE_LORA_PATH else []),
             *([CUSTOM_SPEED_LORA_PATH] if CUSTOM_SPEED_LORA_PATH else [])]:
    if not Path(path).is_file():
        raise FileNotFoundError(path)


## 4. Instalar ComfyUI isolado

Mantém o PyTorch do Kaggle quando ele satisfaz o mínimo do ComfyUI e passa a prova CUDA.
Se for antigo, instala PyTorch 2.8 / CUDA 12.6 no venv. A documentação atual recomenda CUDA 13
em Turing e posteriores; essa opção depende do driver da sessão. Se a prova CUDA falhar,
ajuste `TORCH_INDEX` e as três versões em conjunto para um build compatível com sua GPU/driver.
Não instala SageAttention, xformers nem os nós RunHub personalizados.

Se `ensurepip` falhar no Python da sessão, usa `virtualenv` com pip empacotado e recupera ambientes parciais. O bootstrap fica em uma pasta própria, sem instalar pacotes no Python global.


In [ ]:
if not COMFY.exists():
    COMFY.mkdir()
    run(['git', 'init', COMFY])
    run(['git', '-C', COMFY, 'remote', 'add', 'origin', COMFY_REPO])
    run(['git', '-C', COMFY, 'fetch', '--depth', '1', 'origin', COMFY_COMMIT])
    run(['git', '-C', COMFY, 'checkout', '--detach', COMFY_COMMIT])
else:
    head = subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()
    if head != COMFY_COMMIT:
        raise RuntimeError(f'ComfyUI existente em outro commit ({head}); não será sobrescrito.')
def ensure_python_environment():
    def healthy():
        if not PYTHON.exists():
            return False
        check = subprocess.run(
            [str(PYTHON), '-c', 'import pip,sys; print(sys.prefix)'],
            capture_output=True, text=True,
        )
        return check.returncode == 0 and Path(check.stdout.strip()).resolve() == VENV.resolve()

    if healthy():
        return
    creation = subprocess.run(
        [sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)],
        capture_output=True, text=True,
    )
    if creation.returncode != 0 or not healthy():
        print('venv/ensurepip indisponível ou incompleto; recuperando com virtualenv.')
        bootstrap = ROOT / 'bootstrap'
        pip_probe = subprocess.run(
            [sys.executable, '-m', 'pip', '--version'], capture_output=True, text=True,
        )
        if pip_probe.returncode == 0:
            pip_command = [sys.executable, '-m', 'pip']
        else:
            pip_executable = shutil.which('pip') or shutil.which('pip3')
            if not pip_executable:
                raise RuntimeError('Não há pip disponível na sessão Kaggle. Log do venv: '
                                   + creation.stderr[-2500:])
            pip_command = [pip_executable]
        # Instala o bootstrap numa pasta própria, sem alterar o Python global.
        run([*pip_command, 'install', '--upgrade', '--target', bootstrap, 'virtualenv>=20.26,<21'])
        bootstrap_env = dict(os.environ)
        bootstrap_env['PYTHONPATH'] = str(bootstrap) + os.pathsep + os.environ.get('PYTHONPATH', '')
        # Repara o venv parcial sem apagar ComfyUI, modelos ou arquivos existentes.
        run([sys.executable, '-m', 'virtualenv', '--system-site-packages',
             '--no-download', '--no-periodic-update', VENV], env=bootstrap_env)
    if not healthy():
        raise RuntimeError('O ambiente isolado ainda não possui Python/pip funcionais.')

ensure_python_environment()
run([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
probe = subprocess.run([PYTHON, '-c', "import torch, torchvision, torchaudio; print(torch.__version__.split('+')[0])"], capture_output=True, text=True)
torch_ok = probe.returncode == 0 and tuple(int(x) for x in probe.stdout.strip().split('.')[:2]) >= (2, 7)
if not torch_ok:
    run([PYTHON, '-m', 'pip', 'install', '--index-url', TORCH_INDEX,
         f'torch=={TORCH_VERSION}', f'torchvision=={TORCHVISION_VERSION}', f'torchaudio=={TORCHAUDIO_VERSION}'])
# Evita que um pip install de requirements troque o trio torch e sua variante CUDA.
versions = json.loads(subprocess.check_output([PYTHON, '-c',
    "import json,importlib.metadata as m; print(json.dumps({n:m.version(n) for n in ['torch','torchvision','torchaudio']}))"], text=True))
constraints = ROOT / 'torch-constraints.txt'
constraints.write_text(''.join(f'{name}=={version}\n' for name, version in versions.items()))
run([PYTHON, '-m', 'pip', 'install', '-r', COMFY / 'requirements.txt', '-c', constraints,
     'huggingface_hub>=0.27,<2', 'nbformat>=5.10,<6'])
SERVER_ENV = dict(os.environ, CUDA_VISIBLE_DEVICES=str(GPU_INDEX), HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1')
run([PYTHON, '-c', """import torch, torchvision, torchaudio, comfy_kitchen
assert torch.cuda.is_available(), 'PyTorch não encontra CUDA'
device = torch.cuda.get_device_properties(0)
print('GPU selecionada:', device.name, 'VRAM GiB:', round(device.total_memory / 2**30, 2))
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print('Prova CUDA FP16 OK; torch', torch.__version__, 'CUDA', torch.version.cuda)
print('FP4 nativo disponível:', device.major >= 10)
"""], env=SERVER_ENV)
print('ComfyUI e dependências instalados no venv:', PYTHON)


## 5. Localizar ou baixar somente os pesos necessários

Nomes esperados no Dataset (pastas podem ser diferentes):

| Componente | Arquivo |
|---|---|
| REF2VA pruned | `minimax_h3_ref2va_pruned_int8_convrot.safetensors` |
| Encoder | `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors` |
| VAE vídeo | `minimax_h3_video_vae_int8_convrot.safetensors` |
| VAE áudio | `minimax_h3_audio_vae_fp32.safetensors` |
| Turbo, apenas smoke | `minimax_h3_ref2v_turbo_4step_v0.1_comfyui_bf16.safetensors` |

Fonte: [Comfy-Org/MiniMax-H3](https://huggingface.co/Comfy-Org/MiniMax-H3).
O download resolve `main` para um commit, registra a revisão e verifica tamanho/cabeçalho;
`VERIFY_SHA256=True` ativa a leitura completa e comparação SHA-256. Arquivos de Dataset são ligados
por symlink, sem cópia. Segredos, quando necessários, vêm apenas de Kaggle Secrets / `HF_TOKEN`.


In [ ]:
cfg = settings()
specs = {key: dict(value) for key, value in MODEL_FILES.items()
         if key != 'turbo' or (cfg['turbo'] and not CUSTOM_SPEED_LORA_PATH)}
sources = {key: find_model(key, spec) for key, spec in specs.items()}
missing = [key for key, path in sources.items() if path is None]
if missing and not DOWNLOAD_MISSING_MODELS:
    raise FileNotFoundError('Anexe os modelos ao Dataset: ' + ', '.join(specs[key]['file'] for key in missing))
manifest = {}
if missing or VERIFY_SHA256:
    # Executa com huggingface_hub no venv, sem importar torch no kernel do notebook.
    request = ROOT / 'model-request.json'
    request.write_text(json.dumps({'specs': specs, 'missing': missing,
                                  'verify': VERIFY_SHA256, 'secret_name': HF_SECRET_NAME}))
    downloader = r"""
import json, os, sys, shutil
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
root = Path(sys.argv[1])
req = json.loads((root / 'model-request.json').read_text())
token = os.environ.get('HF_TOKEN')
if req['secret_name']:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(req['secret_name'])
api = HfApi(token=token)
repos = {}
manifest = {}
for key, spec in req['specs'].items():
    if spec['repo'] not in repos:
        repos[spec['repo']] = api.model_info(spec['repo'], files_metadata=True)
    info = repos[spec['repo']]
    sibling = next((s for s in info.siblings if s.rfilename == spec['file']), None)
    if sibling is None or sibling.size is None:
        raise RuntimeError('Arquivo/tamanho ausente no repositório oficial: ' + spec['file'])
    lfs = sibling.lfs
    sha = getattr(lfs, 'sha256', None) if lfs else None
    manifest[key] = dict(repo=spec['repo'], file=spec['file'], revision=info.sha,
                         size=sibling.size, sha256=sha)
needed = sum(manifest[key]['size'] for key in req['missing'])
free = shutil.disk_usage(root).free
if req['missing'] and needed + 5 * 2**30 > free:
    raise RuntimeError(f'Download: {needed/2**30:.1f} GiB + 5 GiB de reserva; livre: {free/2**30:.1f} GiB. '
                       'Anexe os pesos como Dataset em /kaggle/input; nenhum download foi iniciado.')
for key in req['missing']:
    entry = manifest[key]
    path = hf_hub_download(repo_id=entry['repo'], filename=entry['file'], revision=entry['revision'],
                           local_dir=root / 'weights', token=token)
    if Path(path).stat().st_size != entry['size']:
        raise RuntimeError('Download incompleto: ' + entry['file'])
(root / 'download-manifest.json').write_text(json.dumps(manifest, indent=2))
"""
    run([PYTHON, '-c', downloader, ROOT], env=SERVER_ENV)
    manifest = json.loads((ROOT / 'download-manifest.json').read_text())
    sources = {key: find_model(key, spec) for key, spec in specs.items()}
model_names = {}
for key, path in sources.items():
    if path is None:
        raise FileNotFoundError(specs[key]['file'])
    entry = manifest.get(key, {})
    size = safetensors_check(path, entry.get('size'))
    if VERIFY_SHA256:
        expected = entry.get('sha256')
        if not expected:
            raise RuntimeError('SHA-256 publicado indisponível para ' + key)
        if sha256_file(path) != expected:
            raise RuntimeError('SHA-256 diferente do publicado: ' + str(path))
    model_names[key] = link_model(path, specs[key]['folder'], Path(specs[key]['file']).name)
    print(key, f'{size/GiB:.2f} GiB', str(path))
if CUSTOM_SPEED_LORA_PATH:
    model_names['speed_custom'] = link_model(CUSTOM_SPEED_LORA_PATH, 'loras', 'custom_speed.safetensors')
if PEOPLE_LORA_PATH:
    if not 0 <= PEOPLE_STRENGTH <= 2:
        raise ValueError('PEOPLE_STRENGTH deve ficar entre 0 e 2.')
    model_names['people'] = link_model(PEOPLE_LORA_PATH, 'loras', 'people_realism.safetensors')
(ROOT / 'models-used.json').write_text(json.dumps({
    'models': {key: str(path) for key, path in sources.items()}, 'huggingface': manifest,
    'custom_speed': CUSTOM_SPEED_LORA_PATH, 'people': PEOPLE_LORA_PATH,
}, indent=2))
print('Modelos prontos. Os pesos de Dataset permanecem em /kaggle/input.')


## 6. Iniciar servidor local e verificar os nós

Não há interface pública nem túnel. A API local executa a geração e o notebook exibe o MP4.
Se o modo `disk` não for suportado pelo runtime, o erro aparece no log; pode-se testar `classic`,
mas ele move mais trabalho para CPU/RAM e não representa uma solução garantida para OOM.


In [ ]:
import urllib.request, urllib.error

def api(path, payload=None, timeout=30):
    data = json.dumps(payload).encode() if payload is not None else None
    request = urllib.request.Request(BASE_URL + path, data=data,
                                     headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.load(response)
    except urllib.error.HTTPError as error:
        detail = error.read().decode(errors='replace')
        raise RuntimeError(f'ComfyUI HTTP {error.code}: {detail[:6000]}') from error

def server_args():
    args = [str(PYTHON), str(COMFY / 'main.py'), '--listen', '127.0.0.1', '--port', str(PORT),
            '--use-pytorch-cross-attention', '--disable-xformers', '--cache-none',
            '--preview-method', 'none', '--reserve-vram', str(RESERVE_VRAM_GB),
            '--disable-pinned-memory', '--output-directory', str(ROOT / 'output'),
            '--input-directory', str(ROOT / 'input')]
    if CPU_VAE:
        args.append('--cpu-vae')
    if MEMORY_MODE == 'disk':
        args += ['--enable-dynamic-vram', '--fast-disk']
    elif MEMORY_MODE == 'classic':
        args += ['--novram', '--disable-dynamic-vram', '--disable-smart-memory']
    else:
        raise ValueError('MEMORY_MODE deve ser disk ou classic.')
    return args

def log_tail(lines=35):
    if (ROOT / 'comfyui.log').exists():
        return '\n'.join((ROOT / 'comfyui.log').read_text(errors='replace').splitlines()[-lines:])
    return 'Sem log.'

if 'SERVER' in globals() and SERVER.poll() is None:
    queue = api('/queue')
    if queue.get('queue_running') or queue.get('queue_pending'):
        raise RuntimeError('O servidor está gerando. Aguarde ou interrompa sua geração antes de reiniciar.')
    SERVER.terminate()
    try:
        SERVER.wait(timeout=30)
    except subprocess.TimeoutExpired:
        SERVER.kill()
        SERVER.wait(timeout=10)
if 'SERVER_LOG' in globals():
    SERVER_LOG.close()
try:
    api('/system_stats', timeout=2)
except (urllib.error.URLError, TimeoutError, RuntimeError):
    pass
else:
    raise RuntimeError(f'A porta {PORT} está ocupada por outro servidor. Escolha outra PORT na célula 1.')
(ROOT / 'input').mkdir(exist_ok=True)
(ROOT / 'output').mkdir(exist_ok=True)
SERVER_LOG = (ROOT / 'comfyui.log').open('a', buffering=1)
SERVER_LOG.write('\nSTART ' + datetime.now(timezone.utc).isoformat() + '\n')
SERVER = subprocess.Popen(server_args(), cwd=COMFY, env=SERVER_ENV,
                          stdout=SERVER_LOG, stderr=subprocess.STDOUT)
deadline = time.monotonic() + 240
while time.monotonic() < deadline:
    if SERVER.poll() is not None:
        raise RuntimeError('Servidor encerrou na inicialização:\n' + log_tail())
    try:
        stats = api('/system_stats', timeout=3)
        break
    except (urllib.error.URLError, TimeoutError):
        time.sleep(2)
else:
    SERVER.terminate()
    raise TimeoutError('Servidor não respondeu em 4 min:\n' + log_tail())
object_info = api('/object_info')
required_nodes = ['UNETLoader', 'CLIPLoader', 'VAELoader', 'LoadImage',
                  'MiniMaxH3ReferenceToVideo', 'KSamplerSelect', 'BasicScheduler',
                  'BasicGuider', 'RandomNoise', 'SamplerCustomAdvanced',
                  'VAEDecodeTiled', 'VAEDecodeAudio', 'CreateVideo', 'SaveVideo']
if 'turbo' in model_names or 'speed_custom' in model_names or 'people' in model_names:
    required_nodes.append('LoraLoaderModelOnly')
if REFERENCE_AUDIO:
    required_nodes.append('LoadAudio')
absent = [name for name in required_nodes if name not in object_info]
if absent:
    raise RuntimeError('Nós nativos não carregados: ' + ', '.join(absent) + '\n' + log_tail())
print('API e nós nativos disponíveis. Ainda não é um teste de inferência.')
print(log_tail(12))


## 7. Preparar referências e criar o grafo API

As imagens são reduzidas mantendo proporção e EXIF, sem filtros de pele, e recebem nomes próprios
para esta execução. `<Picture 1>`, `<Picture 2>` e `<Picture 3>` seguem a ordem da lista.
O áudio opcional é uma referência de voz/ambiente, não um arquivo que será simplesmente colado ao vídeo.


In [ ]:
def build_prompt(cfg, names, images, text, audio_name=None, people_strength=0.7, seed=42):
    def node(kind, **inputs):
        return {'class_type': kind, 'inputs': inputs}
    graph = {
        'model': node('UNETLoader', unet_name=names['diffusion'], weight_dtype='default'),
        'clip': node('CLIPLoader', clip_name=names['encoder'], type='minimax', device='default'),
        'vvae': node('VAELoader', vae_name=names['video_vae']),
        'avae': node('VAELoader', vae_name=names['audio_vae']),
    }
    model_link = ['model', 0]
    speed = names.get('speed_custom') or (names.get('turbo') if cfg['turbo'] else None)
    if speed:
        graph['speed'] = node('LoraLoaderModelOnly', model=model_link, lora_name=speed, strength_model=1.0)
        model_link = ['speed', 0]
    if 'people' in names:
        graph['people'] = node('LoraLoaderModelOnly', model=model_link, lora_name=names['people'], strength_model=people_strength)
        model_link = ['people', 0]
        if not text.lstrip().startswith('r34l1sm'):
            text = 'r34l1sm\n' + text
    refs = dict(clip=['clip', 0], vae=['vvae', 0], audio_vae=['avae', 0], prompt=text,
                width=cfg['width'], height=cfg['height'], length=cfg['length'], ref_image_size='match')
    for index, image in enumerate(images):
        identifier = f'image_{index}'
        graph[identifier] = node('LoadImage', image=image)
        refs[f'ref_images.ref_image_{index}'] = [identifier, 0]
    if audio_name:
        graph['reference_audio'] = node('LoadAudio', audio=audio_name)
        refs['ref_audios.ref_audio_0'] = ['reference_audio', 0]
    graph.update({
        'condition': node('MiniMaxH3ReferenceToVideo', **refs),
        'guider': node('BasicGuider', model=model_link, conditioning=['condition', 0]),
        'noise': node('RandomNoise', noise_seed=seed),
        'sampler': node('KSamplerSelect', sampler_name='res_multistep'),
        'schedule': node('BasicScheduler', model=model_link, scheduler='beta', steps=cfg['steps'], denoise=1.0),
        'sample': node('SamplerCustomAdvanced', noise=['noise', 0], guider=['guider', 0],
                       sampler=['sampler', 0], sigmas=['schedule', 0], latent_image=['condition', 1]),
        'decode': node('VAEDecodeTiled', samples=['sample', 0], vae=['vvae', 0],
                       tile_size=256, overlap=64, temporal_size=32, temporal_overlap=8),
        'audio': node('VAEDecodeAudio', samples=['sample', 0], vae=['avae', 0]),
        'video': node('CreateVideo', images=['decode', 0], audio=['audio', 0], fps=24.0, bit_depth=8),
        'save': node('SaveVideo', video=['video', 0], filename_prefix='h3_pruned', format='mp4', codec='auto'),
    })
    return graph

cfg = settings()
if not PROMPT.strip():
    raise ValueError('PROMPT está vazio.')
job_label = uuid.uuid4().hex[:12]
prepare_script = r"""
import json, sys, math, shutil
from pathlib import Path
from PIL import Image, ImageOps
import av
root = Path(sys.argv[1]); params = json.loads(sys.argv[2])
names = []
for index, path in enumerate(params['images']):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert('RGB')
        scale = min(1.0, math.sqrt(params['area'] / (image.width * image.height)))
        size = (max(1, round(image.width * scale)), max(1, round(image.height * scale)))
        image = image.resize(size, Image.Resampling.LANCZOS)
        name = f"ref_{params['job']}_{index}.png"
        image.save(root / 'input' / name)
        names.append(name)
audio_name = None
if params['audio']:
    with av.open(params['audio']) as container:
        streams = list(container.streams.audio)
        if not streams:
            raise ValueError('Referência sem faixa de áudio.')
        total = 0.0
        for frame in container.decode(audio=0):
            total += frame.samples / frame.sample_rate
            if total > 15:
                raise ValueError('Referência de áudio excede 15 s.')
        if total <= 0:
            raise ValueError('Referência de áudio vazia.')
    audio_name = 'ref_' + params['job'] + Path(params['audio']).suffix
    shutil.copyfile(params['audio'], root / 'input' / audio_name)
print(json.dumps({'images': names, 'audio': audio_name}))
"""
prepared = json.loads(subprocess.check_output([str(PYTHON), '-c', prepare_script, str(ROOT), json.dumps({
    'images': REFERENCE_IMAGES, 'audio': REFERENCE_AUDIO,
    'area': cfg['width'] * cfg['height'], 'job': job_label,
})], text=True))
graph = build_prompt(cfg, model_names, prepared['images'], PROMPT, prepared['audio'], PEOPLE_STRENGTH, SEED)
# Confirma enums de arquivos e sampler antes de enviar a fila.
for identifier, item in graph.items():
    if item['class_type'] not in object_info:
        raise RuntimeError('Nó ausente: ' + item['class_type'])
    schema = object_info[item['class_type']]['input']
    for key, value in item['inputs'].items():
        definition = schema.get('required', {}).get(key, schema.get('optional', {}).get(key))
        if definition and isinstance(definition[0], list) and not isinstance(value, list):
            if value not in definition[0]:
                raise ValueError(f'{identifier}.{key} não está disponível no servidor: {value}')
(ROOT / 'prompt-api.json').write_text(json.dumps(graph, ensure_ascii=False, indent=2))
(ROOT / 'run-settings.json').write_text(json.dumps({
    'comfy_commit': COMFY_COMMIT, 'profile': cfg, 'seed': SEED,
    'memory_mode': MEMORY_MODE, 'references': REFERENCE_IMAGES,
    'people_strength': PEOPLE_STRENGTH if PEOPLE_LORA_PATH else None,
}, indent=2))
print('Grafo API salvo:', ROOT / 'prompt-api.json')
print('Uma passagem; referência match; resolução:', cfg['width'], cfg['height'])


## 8. Gerar e acompanhar o resultado

A validação do `/prompt` é feita pelo ComfyUI antes da execução. Erros de modelo, quantização ou
memória são exibidos, e não confundidos com vídeo gerado. A primeira execução pode levar bastante tempo
carregando e desquantizando o encoder. Interromper a célula solicita o cancelamento da nossa fila.


In [ ]:
if SERVER.poll() is not None:
    raise RuntimeError('Servidor não está rodando:\n' + log_tail())
queue = api('/queue')
if queue.get('queue_running') or queue.get('queue_pending'):
    raise RuntimeError('Há uma geração em andamento. Aguarde antes de enviar outra.')
response = api('/prompt', {'prompt': graph, 'client_id': uuid.uuid4().hex})
if response.get('node_errors'):
    raise RuntimeError(json.dumps(response['node_errors'], ensure_ascii=False, indent=2))
prompt_id = response['prompt_id']
print('Job:', prompt_id)
deadline = time.monotonic() + GENERATION_TIMEOUT_MINUTES * 60
last_update = 0
try:
    while time.monotonic() < deadline:
        if SERVER.poll() is not None:
            raise RuntimeError('Servidor encerrou; pode ter faltado RAM/VRAM. Exit code: '
                               + str(SERVER.returncode) + '\n' + log_tail())
        history = api('/history/' + prompt_id)
        if prompt_id in history:
            result = history[prompt_id]
            (ROOT / 'last-history.json').write_text(json.dumps(result, ensure_ascii=False, indent=2))
            status = result.get('status', {})
            if status.get('status_str') != 'success' or not status.get('completed'):
                raise RuntimeError('Geração falhou:\n' + json.dumps(status, ensure_ascii=False, indent=2)[-7000:]
                                   + '\n' + log_tail())
            break
        if time.monotonic() - last_update >= 30:
            print(log_tail(3), flush=True)
            last_update = time.monotonic()
        time.sleep(3)
    else:
        api('/interrupt', {})
        raise TimeoutError('Tempo limite alcançado; cancelamento solicitado. Consulte comfyui.log.')
except KeyboardInterrupt:
    api('/interrupt', {})
    print('Cancelamento solicitado ao ComfyUI.')
    raise
output = result.get('outputs', {}).get('save', {})
entries = output.get('images', []) + output.get('videos', []) + output.get('gifs', [])
videos = []
output_root = (ROOT / 'output').resolve()
for entry in entries:
    if entry.get('type') != 'output' or not entry.get('filename', '').endswith('.mp4'):
        continue
    path = (output_root / entry.get('subfolder', '') / entry['filename']).resolve()
    if path.is_relative_to(output_root) and path.is_file() and path.stat().st_size > 0:
        videos.append(path)
if not videos:
    raise RuntimeError('ComfyUI não retornou MP4 válido deste job; não usaremos arquivos de execuções anteriores.')
# Confirma que o MP4 possui frames decodificáveis e áudio, não apenas um nome de arquivo.
verify_video = r"""
import av, json, sys
with av.open(sys.argv[1]) as container:
    if not container.streams.video or not container.streams.audio:
        raise ValueError('MP4 sem vídeo ou áudio.')
    video = container.streams.video[0]
    info = {'width': video.width, 'height': video.height, 'fps': float(video.average_rate)}
    first = next(container.decode(video=0), None)
    if first is None:
        raise ValueError('MP4 sem frames decodificáveis.')
with av.open(sys.argv[1]) as container:
    if next(container.decode(audio=0), None) is None:
        raise ValueError('MP4 sem amostras de áudio decodificáveis.')
print(json.dumps(info))
"""
from IPython.display import Video, display, FileLink
for path in videos:
    info = json.loads(subprocess.check_output([str(PYTHON), '-c', verify_video, str(path)], text=True))
    if (info['width'], info['height']) != (cfg['width'], cfg['height']) or abs(info['fps'] - 24) > 0.01:
        raise RuntimeError('Resolução/FPS do MP4 diferente do solicitado: ' + str(info))
    print('MP4 verificado:', path, info)
    display(Video(str(path), embed=True))
    display(FileLink(str(path)))


## Problemas comuns e próximos testes

- **Disco insuficiente:** os pesos ficam perto de 42 GB. Anexe-os como Dataset; reduzir frames não reduz o tamanho dos pesos.
- **CUDA indisponível / no kernel image / driver insuficiente:** confirme o Accelerator e um trio torch/vision/audio compatível;
  consulte a prova CUDA da célula 4. CUDA 13 requer driver compatível. Não instale atenção compilada aleatoriamente.
- **Killed / exit 137 / OOM de RAM:** `classic` pode piorar porque usa CPU. Com modelos em Dataset,
  tente `disk`, uma imagem e `smoke`; se ainda falhar, essa sessão não comporta o modelo/encoder.
  Pruning sozinho não elimina o Qwen3-VL 32B nem a memória temporária de desquantização.
- **OOM de VRAM:** mantenha `CPU_VAE=True`, aumente `RESERVE_VRAM_GB`, use uma imagem e o perfil `smoke`.
  Duas T4 não somam memória automaticamente; não existe uma opção de VRAM virtual aqui.
- **Pele artificial:** depois de um teste bem-sucedido, use `qualidade` sem turbo, close-up e luz natural;
  opcionalmente teste sua LoRA People com peso moderado. Resolução menor sacrifica detalhes de poros.
- **LoRA incompatível:** desligue People / custom speed e valide primeiro o modelo base.
- **401/403 no Hugging Face:** verifique acesso ao modelo e Internet; quando necessário, use Kaggle Secrets.
- **Após reiniciar a sessão:** os arquivos de `/kaggle/working` podem desaparecer; anexe modelos como Dataset
  e salve/download os MP4. Os links do notebook não publicam o servidor.

Diagnóstico: `comfyui.log`, `prompt-api.json`, `models-used.json`, `run-settings.json`, `last-history.json`.
O JSON API é uma adaptação executável, não o layout visual original. O JSON original acompanha o repositório
como referência, mas seus links de prévia, notas e instruções não são executados por este notebook.

Fontes consultadas:

- [ComfyUI v0.32.0](https://github.com/Comfy-Org/ComfyUI/tree/c2bcbecd82ec5ae66594340b395c24ef0217b238)
- [Template oficial REF2VA consultado](https://github.com/Comfy-Org/workflow_templates/blob/b4119d0fcecf78a376e087305ec4f485c66a03f5/templates/video_minimax_h3_r2v.json)
- [Modelos oficiais H3](https://huggingface.co/Comfy-Org/MiniMax-H3)


In [ ]:
# Opcional: execute esta célula manualmente depois de terminar; Run All não para o servidor.
STOP_SERVER = False
if STOP_SERVER and 'SERVER' in globals() and SERVER.poll() is None:
    SERVER.terminate()
    try:
        SERVER.wait(timeout=30)
    except subprocess.TimeoutExpired:
        SERVER.kill()
        SERVER.wait(timeout=10)
    SERVER_LOG.close()
    print('Servidor encerrado.')
